# WOOOHOOO OFFICIAL SCRAPER

In [2]:
# ── Single Cell — Amazon Best Sellers Scraper (Multi-Department) ──────────────
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from IPython.display import display
import pandas as pd
import time, json, random

# ── 1. Departments ────────────────────────────────────────────────────────────
DEPARTMENTS = {
    "Electronics":               "https://www.amazon.com/Best-Sellers-Electronics/zgbs/electronics/",
    "Clothing, Shoes & Jewelry": "https://www.amazon.com/Best-Sellers-Clothing-Shoes-Jewelry/zgbs/fashion/",
    "Kitchen & Home":            "https://www.amazon.com/Best-Sellers-Kitchen-Dining/zgbs/kitchen/",
}

# ── 2. Driver ─────────────────────────────────────────────────────────────────
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
    "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
})
wait = WebDriverWait(driver, 20)

# ── 3. Helpers ────────────────────────────────────────────────────────────────
def get_text(ctx, selector, default="N/A"):
    try: return ctx.find_element(By.CSS_SELECTOR, selector).text.strip()
    except: return default

def get_first_text(ctx, selectors, default="N/A"):
    for sel in selectors:
        try:
            val = ctx.find_element(By.CSS_SELECTOR, sel).text.strip()
            if val: return val
        except: continue
    return default

def human_sleep(a=2, b=4):
    time.sleep(random.uniform(a, b))

def check_for_captcha(driver):
    if "captcha" in driver.page_source.lower() or "Type the characters" in driver.page_source:
        print("⚠️  CAPTCHA detected! Solve it in the browser then press Enter...")
        input()

def product_check(product):
    REQUIRED = ["name", "description", "brand", "price", "rating_score", "rating_count", "availability"]
    print(f"\n  📋 Field Check:")
    for field in REQUIRED:
        val = product.get(field, "N/A")
        status = "✅" if val and val != "N/A" else "❌"
        print(f"     {status} {field:20s}: {str(val)[:60] if status == '✅' else 'N/A'}")

def review_check(reviews):
    if not reviews: return
    last = reviews[-1]
    problems = [f for f, v in last.items()
                if v in ("N/A", "", None) and f not in ("images", "helpful_votes", "variant")]
    if problems:
        print(f"     ⚠️  Review #{len(reviews)} N/A: {problems} | title='{last['title']}' | body='{str(last['body'])[:40]}'")

def get_product_urls(driver):
    for _ in range(10):          # more scrolls to reach all 50 items
        driver.execute_script("window.scrollBy(0, 800);")
        time.sleep(0.8)          # longer wait between scrolls for lazy loading
    time.sleep(2)                # extra wait at the end
    links = driver.find_elements(By.CSS_SELECTOR, "a[href*='/dp/']")
    urls, seen = [], set()
    for el in links:
        href = el.get_attribute("href") or ""
        if "/dp/" in href:
            try:
                asin = href.split("/dp/")[1].split("/")[0].split("?")[0]
                if asin not in seen and len(asin) == 10:
                    seen.add(asin)
                    urls.append(f"https://www.amazon.com/dp/{asin}")
            except: continue
    return urls[:50]             # cap at 50

def scrape_product(driver, url, index, department):
    driver.get(url)
    human_sleep(2, 4)
    check_for_captcha(driver)

    # Scroll slowly to trigger all lazy-loaded content
    for _ in range(12):
        driver.execute_script("window.scrollBy(0, 500);")
        time.sleep(0.6)
    time.sleep(2)
    driver.execute_script("window.scrollTo(0, 0);")
    human_sleep(1, 2)

    # ── Description via JS ────────────────────────────────────────────────────
    description = driver.execute_script("""
        var el = document.querySelector('#productDescription p span');
        if (!el) el = document.querySelector('#productDescription p');
        if (!el) el = document.querySelector('#productDescription');
        return el ? (el.innerText || el.textContent || '').trim() : null;
    """) or None

    # Fallback — "About this item" bullets if description is empty
    if not description:
        try:
            bullets = driver.find_elements(By.CSS_SELECTOR, "#feature-bullets li span.a-list-item")
            bullet_texts = [b.text.strip() for b in bullets if b.text.strip()]
            description = " | ".join(bullet_texts) if bullet_texts else "N/A"
        except:
            description = "N/A"

    # ── Price via JS (handles visually hidden .a-offscreen spans) ─────────────
    price = driver.execute_script("""
        var selectors = [
            '#corePrice_feature_div .a-offscreen',
            '#corePriceDisplay_desktop_feature_div .a-price-whole',
            '#apex_offerDisplay_desktop .a-price .a-offscreen',
            '.a-price .a-offscreen',
        ];
        for (var s of selectors) {
            var el = document.querySelector(s);
            if (el) {
                var val = (el.innerText || el.innerHTML || '').trim();
                if (val) return val;
            }
        }
        return 'N/A';
    """) or "N/A"

    # ── Product info ──────────────────────────────────────────────────────────
    product = {
        "department":         department,
        "index":              index,
        "url":                driver.current_url,
        "name":               get_text(driver, "#productTitle"),
        "description":        description,
        "brand":              get_text(driver, "#bylineInfo"),
        "rating_score":       get_text(driver, "[data-hook='rating-out-of-text']"),
        "rating_count":       get_text(driver, "#acrCustomerReviewText"),
        "price":              price,
        "availability": get_first_text(driver, [
            "#availability span",
            "#availabilityInsideBuyBox_feature_div span",
            "#deliveryBlockMessage span",
        ]),
        "answered_questions": get_text(driver, "#askATFLink span"),
    }

    product_check(product)

    # ── Highlights ────────────────────────────────────────────────────────────
    try:
        bullets = driver.find_elements(By.CSS_SELECTOR, "#feature-bullets li span.a-list-item")
        product["highlights"] = [b.text.strip() for b in bullets if b.text.strip()]
    except:
        product["highlights"] = []

    # ── Variants ──────────────────────────────────────────────────────────────
    try:
        variant_groups = driver.find_elements(By.CSS_SELECTOR, ".a-section.a-spacing-small.poi-rotate-anchor")
        variants = {}
        for grp in variant_groups:
            label = get_text(grp, "label.a-form-label")
            opts  = [o.get_attribute("title") or o.text for o in grp.find_elements(By.CSS_SELECTOR, "li")]
            opts  = [o.strip() for o in opts if o and o.strip()]
            if label and opts:
                variants[label.rstrip(":")] = opts
        product["variants"] = variants
    except:
        product["variants"] = {}

    # ── Images ────────────────────────────────────────────────────────────────
    try:
        thumbs = driver.find_elements(By.CSS_SELECTOR, "#altImages img")
        product["images"] = list({img.get_attribute("src") for img in thumbs if img.get_attribute("src")})
    except:
        product["images"] = []

    # ── Specs ─────────────────────────────────────────────────────────────────
    try:
        rows = driver.find_elements(By.CSS_SELECTOR,
            "#productDetails_techSpec_section_1 tr, #productDetails_detailBullets_sections1 tr")
        specs = {}
        for row in rows:
            cols = row.find_elements(By.TAG_NAME, "td")
            if len(cols) >= 2:
                specs[cols[0].text.strip()] = cols[1].text.strip()
        if not specs:
            for item in driver.find_elements(By.CSS_SELECTOR, "#detailBullets_feature_div li"):
                text = item.text.strip()
                if ":" in text:
                    k, v = text.split(":", 1)
                    specs[k.strip().strip("\u200f")] = v.strip()
        product["specifications"] = specs
    except:
        product["specifications"] = {}

    # ── Reviews (on-page only) ────────────────────────────────────────────────
    reviews = []
    cards = driver.find_elements(By.CSS_SELECTOR, "[data-hook='review']")

    for card in cards:
        def g(sel, default="N/A"):
            for s in sel.split(" || "):
                try:
                    val = card.find_element(By.CSS_SELECTOR, s.strip()).text.strip()
                    if val: return val
                except: continue
            return default

        try:
            star_el = card.find_element(By.CSS_SELECTOR,
                "[data-hook='review-star-rating'] span, [data-hook='cmps-review-star-rating'] span")
            rating = star_el.get_attribute("innerHTML").split(" ")[0]
        except:
            rating = "N/A"

        try:
            imgs   = card.find_elements(By.CSS_SELECTOR, "[data-hook='review-image-tile']")
            images = [i.get_attribute("src") for i in imgs if i.get_attribute("src")]
        except:
            images = []

        try:
            body_els = card.find_elements(By.CSS_SELECTOR, "[class*='review-text'] span")
            body = " ".join([el.text.strip() for el in body_els if el.text.strip()]) or "N/A"
        except:
            body = "N/A"

        reviews.append({
            "reviewer":      g(".a-profile-name"),
            "rating":        rating,
            "date":          g("[data-hook='review-date']"),
            "verified":      g("[data-hook='avp-badge']"),
            "title":         g("a[data-hook='review-title'] h5 || h5[data-hook='reviewTitle'] || [data-hook='review-title']"),
            "body":          body,
            "helpful_votes": g("[data-hook='helpful-vote-statement']"),
            "variant":       g("[data-hook='format-strip']"),
            "image_count":   len(images),
            "images":        images,
        })

        review_check(reviews)

    product["reviews"] = reviews
    return product

# ── 4. Main loop ──────────────────────────────────────────────────────────────
all_products     = []
all_reviews_flat = []

try:
    for dept_name, dept_url in DEPARTMENTS.items():
        print(f"\n{'═'*60}")
        print(f"🏬 Department: {dept_name}")
        print(f"{'═'*60}")

        driver.get(dept_url)
        human_sleep(3, 5)
        check_for_captcha(driver)

        product_urls = get_product_urls(driver)
        print(f"🔍 Found {len(product_urls)} products (capped at 50)")

        for i, url in enumerate(product_urls):
            print(f"\n{'─'*60}")
            print(f"📦 [{i+1}/{len(product_urls)}] {url}")
            try:
                product = scrape_product(driver, url, i + 1, dept_name)
                all_products.append(product)
                print(f"   ✅ {product['name'][:60]}")
                print(f"   💬 {len(product['reviews'])} reviews")

                for r in product.get("reviews", []):
                    all_reviews_flat.append({
                        "department":    dept_name,
                        "product_index": i + 1,
                        "product_name":  product.get("name", "N/A"),
                        "product_url":   url,
                        **{k: v for k, v in r.items() if k != "images"},
                    })
            except Exception as e:
                print(f"   ❌ Failed: {e}")
                all_products.append({
                    "department": dept_name, "index": i + 1,
                    "url": url, "error": str(e), "reviews": []
                })

            human_sleep(2, 4)

    # ── 5. Save ───────────────────────────────────────────────────────────────
    with open("amazon_bestsellers.json", "w", encoding="utf-8") as f:
        json.dump(all_products, f, ensure_ascii=False, indent=2)

    df = pd.DataFrame(all_reviews_flat)
    df.to_csv("amazon_bestsellers_reviews.csv", index=False)
    print(f"\n💾 Saved: amazon_bestsellers.json + amazon_bestsellers_reviews.csv ({len(df)} reviews)")

    summary = pd.DataFrame([{
        "department":      p.get("department"),
        "index":           p.get("index"),
        "name":            p.get("name", "N/A")[:45],
        "price":           p.get("price", "N/A"),
        "rating_score":    p.get("rating_score", "N/A"),
        "rating_count":    p.get("rating_count", "N/A"),
        "reviews_scraped": len(p.get("reviews", [])),
    } for p in all_products])

    print(f"\n📊 Summary ({len(all_products)} products across {len(DEPARTMENTS)} departments):")
    display(summary)
    display(df.head(10))

except Exception as e:
    import traceback
    print(f"❌ Error: {e}")
    traceback.print_exc()
    print(f"\n🔍 URL  : {driver.current_url}")
    print(f"🔍 Title: {driver.title}")

finally:
    time.sleep(2)
    driver.quit()
    print("🛑 Driver closed")


════════════════════════════════════════════════════════════
🏬 Department: Electronics
════════════════════════════════════════════════════════════
🔍 Found 50 products (capped at 50)

────────────────────────────────────────────────────────────
📦 [1/50] https://www.amazon.com/dp/B08JHCVHTY

  📋 Field Check:
     ❌ name                : N/A
     ❌ description         : N/A
     ❌ brand               : N/A
     ❌ price               : N/A
     ❌ rating_score        : N/A
     ❌ rating_count        : N/A
     ❌ availability        : N/A
   ✅ N/A
   💬 0 reviews

────────────────────────────────────────────────────────────
📦 [2/50] https://www.amazon.com/dp/B0DCH8VDXF

  📋 Field Check:
     ✅ name                : Apple EarPods Headphones with USB-C Plug, Wired Ear Buds wit
     ✅ description         : SUPERIOR COMFORT — Unlike traditional circular ear buds, the
     ✅ brand               : Visit the Apple Store
     ❌ price               : N/A
     ✅ rating_score        : 4.6 out of 5
   

,department,index,name,price,rating_score,rating_count,reviews_scraped
0,Electronics,1,N/A,N/A,N/A,N/A,0
1,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wir",N/A,4.6 out of 5,"(12,872)",13
2,Electronics,3,Apple AirTag (2nd Generation): Tracker for Ke,$20.99,4.6 out of 5,"(2,828)",8
3,Electronics,4,"Apple AirPods 4 Wireless Earbuds, Bluetooth H",N/A,4.6 out of 5,"(28,569)",8
4,Electronics,5,"Apple AirPods Pro 3 Wireless Earbuds, Active",N/A,4.5 out of 5,"(8,779)",8
...,...,...,...,...,...,...,...
145,Kitchen & Home,46,Amazon Basics Non-Stick Parchment Paper for B,$6.94,4.7 out of 5,"(7,363)",8
146,Kitchen & Home,47,HOTOR Insulated Lunch Box for Men & Women - L,$18.69,4.5 out of 5,"(6,296)",8
147,Kitchen & Home,48,Instant Pot Duo 7-in-1 Electric Pressure Cook,$96.13,4.7 out of 5,"(184,159)",13
148,Kitchen & Home,49,REDUCE Saltini 16 oz Cocktail Tumbler - Insul,$24.99,4.7 out of 5,(975),13


,department,product_index,product_name,product_url,reviewer,rating,date,verified,title,body,helpful_votes,variant,image_count
0,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Sridhar Mani,5,"Reviewed in the United States on March 17, 2026",Verified Purchase,Best-in-Class Quality and Sound,These Apple EarPods with USB-C are truly best ...,28 people found this helpful,Size: One SizeStyle: USB-C,0
1,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,A.S,5,"Reviewed in the United States on April 11, 2026",Verified Purchase,Great Sound and Comfort,"I bought these for my son, who prefers traditi...",8 people found this helpful,Size: One SizeStyle: USB-C,0
2,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Amazon_Customer,5,"Reviewed in the United States on April 10, 2026",Verified Purchase,"The $9 Audiophile Secret: Small Dongle, Big Pr...",Let’s be honest: nobody actually wants to live...,7 people found this helpful,Size: One SizeStyle: USB-C,0
3,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,K,5,"Reviewed in the United States on March 17, 2026",Verified Purchase,better than overhead headphones,I honestly love these headphones so much. I ac...,23 people found this helpful,Size: One SizeStyle: USB-C,0
4,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Low,5,"Reviewed in the United States on April 25, 2026",Verified Purchase,Worth the extra money. They're the best ones I...,"Best you can buy. I have 4 usbc ear buds, thes...",N/A,Size: One SizeStyle: USB-C,0
5,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,MIHOKO REIWA,4,"Reviewed in the United States on February 22, ...",Verified Purchase,Fit issues,If you’ve used an iPhone from before the USB-C...,21 people found this helpful,Size: One SizeStyle: USB-C,1
6,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Gio Arenas,5,"Reviewed in the United States on March 23, 2026",Verified Purchase,"My Thoughts As a Samsung ""Fanboy""","Like the title says, I'm a Samsung guy through...",18 people found this helpful,Size: One SizeStyle: USB-C,0
7,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Johan Marin,5,"Reviewed in the United States on April 14, 2026",Verified Purchase,Great product,"Great product, exactly what I expected. The so...",2 people found this helpful,Size: One SizeStyle: USB-C,0
8,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,LuckyLucy,5,"Reviewed in Canada on December 20, 2025",Verified Purchase,Best... BEST Budget wired USBC earbuds .,Apple EarPods (USB-C) Okay honestly for $24 th...,N/A,Size: One SizeStyle: USB-C,1
9,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Jose Luis Merino,5,"Reviewed in Mexico on April 22, 2025",Verified Purchase,"Clásicos, confiables y actualizados",Los Apple EarPods con conector USB-C represent...,N/A,Size: One SizeStyle: USB-C,0


🛑 Driver closed
